<a href="https://colab.research.google.com/github/viettungvuong/Distillation-Playground/blob/main/Distillation_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install peft bitsandbytes accelerate flash-attn

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done


**Huggingface login**

In [ ]:
from huggingface_hub import login

login()

**Downloading dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("OpenHust/vietnamese-summarization")

In [ ]:
from transformers import AutoProcessor

model_name = "Qwen/Qwen3.5-9B-Base"
processor = AutoProcessor.from_pretrained(model_name)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
import torch

# 1. Load model in native bfloat16 with Flash Attention 2 (no 4-bit quantization needed for 80GB VRAM)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map={(""): 0},
    trust_remote_code=True
)

model.gradient_checkpointing_enable()

# 2. Define LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

**Data processing**

In [ ]:
dataset

In [ ]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)

train_set = split_dataset['train']
test_set = split_dataset['test']

In [ ]:
def normalize_text(row):
    # Only handle lowercase normalization
    row['Document'] = str(row['Document']).lower()
    row['Summary'] = str(row['Summary']).lower()
    return row

def add_special_tokens(row):
    # Qwen follows the ChatML format:
    # <|im_start|>user
    # {Document}<|im_end|>
    # <|im_start|>assistant
    # {Summary}<|im_end|>

    formatted_input = (
        f"<|im_start|>\n"
        f"{row['Document']}<|im_end|>\n"
        f"<|im_start|>\n"
        f"{row['Summary']}<|im_end|>"
    )

    row['Processed_Input'] = formatted_input
    return row

def process(row):
    normalized_row = normalize_text(row)
    return add_special_tokens(normalized_row)

train_set = train_set.map(process)
test_set = test_set.map(process)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
print("Special tokens: ", tokenizer.special_tokens_map)
sep_token_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
print(f"Sep token is {sep_token_id}")

In [ ]:
def tokenize_function(batch):
    # Tokenize the processed input
    model_inputs = tokenizer(
        batch["Processed_Input"],
        truncation=True,
        max_length=1542,
        padding='max_length'
    )

    labels = []

    for input_ids in model_inputs["input_ids"]:
        label = list(input_ids)
        try:
            # Find the index of the first separation token
            indices = [i for i, val in enumerate(label) if val == sep_token_id]
            turn_model_idx = indices[-1]
            # Set all tokens up to the first separation to -100 (loss function ignore)
            for i in range(turn_model_idx + 1):
                label[i] = -100
        except ValueError as e:
            print(e)

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs


In [ ]:
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=train_set.column_names)
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=test_set.column_names)

**Fine-tuning**

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="summarization",
    num_train_epochs=5,
    per_device_train_batch_size=16, # Increased since we have 80GB VRAM
    gradient_accumulation_steps=4,  # Adjusted to maintain an effective batch size of 64
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    optim="adamw_torch"
)

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
# Re-initialize trainer to pick up the LoRA wrapped model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
trainer.train()